In [50]:
from collections import Counter
from copy import deepcopy
from functools import total_ordering
from itertools import islice
from operator import attrgetter, itemgetter
from typing import Iterable
import math

import dask.bag as db
import toolz
from IPython.display import display

In [310]:
def key_func(*args, **kwargs):
    args = args[0]
    _args = [hash(args[0])]
    for arg in args[1:]:
        if isinstance(arg, list):
            _args.append(tuple(arg))
        elif isinstance(arg, slice):
            _args.append((arg.start, arg.stop, arg.step))
        else:
            _args.append(arg)
    key = tuple(_args)
    if kwargs:
        key = key + tuple(sorted(kwargs.items()))
    return key

@total_ordering
class Num:

    def __init__(self, num) -> None:
        self.num = num
        self.size = abs(num)
        self.sign = 1 if num >= 0 else -1
        self.range = range(0, self.num, self.sign)
        self.reset()

    def reset(self):
        self.iter = self._get_iter()

    def _get_iter(self):
        return iter(map(type(self), self.range))

    def __iter__(self):
        return self._get_iter()

    def __next__(self):
        try:
            out = next(self.iter)
            if out == self.num - self.sign:
                self.reset()
            return out
        except StopIteration:
            self.reset()
            raise

    def __repr__(self) -> str:
        return f"Num({self.num})"

    def __str__(self) -> str:
        return str(self.num)
    
    def __len__(self) -> int:
        return self.size

    def __hash__(self) -> int:
        return hash(self.num) + hash(type(self))

    def _other(self, other):
        if hasattr(other, "num"):
            other = other.num
        return other

    def __eq__(self, other):
        return self.num == self._other(other)

    def __lt__(self, other):
        return self.num < self._other(other)

    def __contains__(self, item):
        return item in self.range

    def __bool__(self):
        return bool(self.num)

    @toolz.functoolz.memoize(key=key_func)
    def __getitem__(self, index):
        """Get item, items, or slice of items, accepts int, list, or slice."""
        if isinstance(index, int):
            if index < 0:
                index = self.size + index
            if index >= self.size or index < 0:
                return None
            return toolz.nth(index, self._get_iter())
        elif isinstance(index, Iterable):
            return type(index)(toolz.pluck(index, (self,)))
        elif isinstance(index, slice):
            return tuple(islice(self, index.start, index.stop, index.step))
        else:
            raise TypeError(f"Invalid index type: {type(index)}")

    def _convert(self, result):
        if isinstance(result, (type(self), Iterable)):
            return result
        try:
            _result = int(result)
            if _result == result:
                result = _result
        except TypeError:
            pass
        try:
            return type(self)(result)
        except TypeError:
            return result

    def __add__(self, other):
        return self._convert(self.num + self._other(other))

    def __sub__(self, other):
        return self._convert(self.num - self._other(other))

    def __mul__(self, other):
        return self._convert(self.num * self._other(other))

    def __floordiv__(self, other):
        return self._convert(self.num // self._other(other))

    def __truediv__(self, other):
        return self._convert(self.num / self._other(other))

    def __mod__(self, other):
        return self._convert(self.num % self._other(other))

    def __pow__(self, other):
        return self._convert(self.num ** self._other(other))

    def __lshift__(self, other):
        return self._convert(self.num << self._other(other))

    def __rshift__(self, other):
        return self._convert(self.num >> self._other(other))

    def __and__(self, other):
        return self._convert(self.num & self._other(other))

    def __xor__(self, other):
        return self._convert(self.num ^ self._other(other))

    def __or__(self, other):
        return self._convert(self.num | self._other(other))

    def __radd__(self, other):
        return type(other)(self._other(other) + self.num)

    def __rsub__(self, other):
        return type(other)(self._other(other) - self.num)

    def __rmul__(self, other):
        return type(other)(self._other(other) * self.num)

    def __rfloordiv__(self, other):
        return type(other)(self._other(other) // self.num)

    def __rtruediv__(self, other):
        return type(other)(self._other(other) / self.num)

    def __rmod__(self, other):
        return type(other)(self._other(other) % self.num)

    def __rpow__(self, other):
        return type(other)(self._other(other) ** self.num)

    def __rlshift__(self, other):
        return type(other)(self._other(other) << self.num)

    def __rrshift__(self, other):
        return type(other)(self._other(other) >> self.num)

    def __rand__(self, other):
        return type(other)(self._other(other) & self.num)

    def __rxor__(self, other):
        return type(other)(self._other(other) ^ self.num)

    def __ror__(self, other):
        return type(other)(self._other(other) | self.num)

    def __iadd__(self, other):
        self.num += self._other(other)
        return self

    def __isub__(self, other):
        self.num -= self._other(other)
        return self

    def __imul__(self, other):
        self.num *= self._other(other)
        return self

    def __ifloordiv__(self, other):
        self.num //= self._other(other)
        return self

    def __itruediv__(self, other):
        self.num /= self._other(other)
        return self

    def __imod__(self, other):
        self.num %= self._other(other)
        return self

    def __ipow__(self, other):
        self.num **= self._other(other)
        return self

    def __ilshift__(self, other):
        self.num <<= self._other(other)
        return self

    def __irshift__(self, other):
        self.num >>= self._other(other)
        return self

    def __iand__(self, other):
        self.num &= self._other(other)
        return self

    def __ixor__(self, other):
        self.num ^= self._other(other)
        return self

    def __ior__(self, other):
        self.num |= self._other(other)
        return self

    def __neg__(self):
        return self._convert(-self.num)

    def __pos__(self):
        return self._convert(+self.num)

    def __abs__(self):
        return self._convert(self.size)

    def __invert__(self):
        return self._convert(~self.num)

    def __int__(self):
        return int(self.num)

    def __float__(self):
        return float(self.num)

    def __complex__(self):
        return complex(self.num)

    def __round__(self, n=None):
        return self._convert(round(self.num, n))

    def __floor__(self):
        return self._convert(math.floor(self.num))

    def __ceil__(self):
        return self._convert(math.ceil(self.num))

    def __trunc__(self):
        return self._convert(math.trunc(self.num))

    def __index__(self):
        return int(self)

    def __matmul__(self, other):
        return [list(self)] * self._other(other)

nums = list(map(Num, range(10)))
print(nums)
print(Num(10))
print(Num(10)[5])
print(Num(10)[[2, 3]])
print(Num(10)[slice(2, 5)])

print(Num(5) @ Num(3))
print(list("abc")[Num(1)])
print(Num(2) + 2j)

for _ in Num(3):
    print(repr(_))

num = Num(10)
print(num)
print(num[5])
print(num[[2, 3]])
print(num[slice(2, 5)])
print(list(num))

for _ in range(3):
    print(repr(_), next(num))

for _ in num:
    print(repr(_))
    break

for _ in num:
    print(repr(_))
    break

print(list(num))

[Num(0), Num(1), Num(2), Num(3), Num(4), Num(5), Num(6), Num(7), Num(8), Num(9)]
10
5
[(Num(2), Num(3))]
(Num(2), Num(3), Num(4))
[[Num(0), Num(1), Num(2), Num(3), Num(4)], [Num(0), Num(1), Num(2), Num(3), Num(4)], [Num(0), Num(1), Num(2), Num(3), Num(4)]]
b
(2+2j)
Num(0)
Num(1)
Num(2)
10
5
[(Num(2), Num(3))]
(Num(2), Num(3), Num(4))
[Num(0), Num(1), Num(2), Num(3), Num(4), Num(5), Num(6), Num(7), Num(8), Num(9)]
0 0
1 1
2 2
Num(0)
Num(0)
[Num(0), Num(1), Num(2), Num(3), Num(4), Num(5), Num(6), Num(7), Num(8), Num(9)]


In [262]:
from operator import methodcaller


def summer(value, *args):
    for _ in args:
        value = value + _
    return value

def counter(items):
    return Counter(items).items()

def sum_counts(values):
    value, *args = values
    value = list(value)
    if len(value) == 1:
        value = value[0]
    value = Counter(dict(value))
    for _ in args:
        value.update(dict(_))
    return value

class ProxyObjects:
    items: db.Bag

    def __init__(self, items, **kwargs):
        if isinstance(items, db.Bag):
            self.items = items
            npartitions = kwargs.pop('npartitions', None)
            if npartitions is not None:
                self.items = self.items.repartition(npartitions)
        else:
            self.items = db.from_sequence(items, **kwargs)

    def __getattr__(self, attr):
        func = attrgetter(attr)
        if hasattr(self.items, attr):
            return func(self.items)
        return ProxyObjects(self.items.map(func))
    
    def __getitem__(self, item):
        return ProxyObjects(self.pluck(item))
    
    def __call__(self, *args, **kwargs):
        return ProxyObjects(self.items.map(lambda x: x(*args, **kwargs)))
    
    def call(self, method, *args, **kwargs):
        func = methodcaller(method, *args, **kwargs)
        return ProxyObjects(self.items.map(func))

    def map(self, func, *args, compute=False, **kwargs):
        out = ProxyObjects(self.items.map(func, *args, **kwargs))
        if compute:
            return out.compute()
        return out
    
    def compute(self, *args, flatten=False, **kwargs):
        items = self.items
        if flatten:
            items = items.flatten()
        return items.compute(*args, **kwargs)
    
    def filter(self, func, *args, **kwargs):
        return ProxyObjects(self.items.filter(func, *args, **kwargs))
    
    def flatten(self):
        return ProxyObjects(self.items.flatten())

    def __repr__(self):
        return self.items.__repr__()
    
    def __str__(self):
        return self.items.__str__()
    
    def reduction(self, *args, **kwargs):
        return self.items.reduction(*args, **kwargs)

    def counts(self, split_every=None):
        return self.reduction(counter, sum_counts, split_every=split_every).compute()

In [266]:
nums = list(Num(20))  # + list(Num(10)) + list(Num(5)) + list(Num(1)) + list(Num(0))
objs = ProxyObjects(nums, npartitions=2)
print(repr(objs))

dask.bag<from_sequence, npartitions=2>


In [219]:
complex_nums = objs.__complex__()
complex_nums.compute()

[0j,
 (1+0j),
 (2+0j),
 (3+0j),
 (4+0j),
 (5+0j),
 (6+0j),
 (7+0j),
 (8+0j),
 (9+0j),
 (10+0j),
 (11+0j),
 (12+0j),
 (13+0j),
 (14+0j),
 (15+0j),
 (16+0j),
 (17+0j),
 (18+0j),
 (19+0j)]

In [144]:
objs.call("__lt__", 10).compute()

[True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 False,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True,
 True]

In [147]:
objs.filter(lambda x: x % 5 == 0).filter(lambda x: len(x)).compute?

Object `compute` not found.


In [42]:
for _ in nums:
    if _ % 5 == 0:
        print(repr(_))

Num(0)
Num(5)
Num(10)
Num(15)


In [43]:
db.map(lambda x: x + 1, objs.items).compute()

[[Num(0)],
 [Num(0), Num(1)],
 [Num(0), Num(1), Num(2)],
 [Num(0), Num(1), Num(2), Num(3)],
 [Num(0), Num(1), Num(2), Num(3), Num(4)],
 [Num(0), Num(1), Num(2), Num(3), Num(4), Num(5)],
 [Num(0), Num(1), Num(2), Num(3), Num(4), Num(5), Num(6)],
 [Num(0), Num(1), Num(2), Num(3), Num(4), Num(5), Num(6), Num(7)],
 [Num(0), Num(1), Num(2), Num(3), Num(4), Num(5), Num(6), Num(7), Num(8)],
 [Num(0),
  Num(1),
  Num(2),
  Num(3),
  Num(4),
  Num(5),
  Num(6),
  Num(7),
  Num(8),
  Num(9)],
 [Num(0),
  Num(1),
  Num(2),
  Num(3),
  Num(4),
  Num(5),
  Num(6),
  Num(7),
  Num(8),
  Num(9),
  Num(10)],
 [Num(0),
  Num(1),
  Num(2),
  Num(3),
  Num(4),
  Num(5),
  Num(6),
  Num(7),
  Num(8),
  Num(9),
  Num(10),
  Num(11)],
 [Num(0),
  Num(1),
  Num(2),
  Num(3),
  Num(4),
  Num(5),
  Num(6),
  Num(7),
  Num(8),
  Num(9),
  Num(10),
  Num(11),
  Num(12)],
 [Num(0),
  Num(1),
  Num(2),
  Num(3),
  Num(4),
  Num(5),
  Num(6),
  Num(7),
  Num(8),
  Num(9),
  Num(10),
  Num(11),
  Num(12),
  Num(13)]

In [222]:
inverted = objs.__invert__()
inverted

dask.bag<lambda, npartitions=2>

In [221]:
inverted.compute()

[[Num(0)],
 [Num(0), Num(-1)],
 [Num(0), Num(-1), Num(-2)],
 [Num(0), Num(-1), Num(-2), Num(-3)],
 [Num(0), Num(-1), Num(-2), Num(-3), Num(-4)],
 [Num(0), Num(-1), Num(-2), Num(-3), Num(-4), Num(-5)],
 [Num(0), Num(-1), Num(-2), Num(-3), Num(-4), Num(-5), Num(-6)],
 [Num(0), Num(-1), Num(-2), Num(-3), Num(-4), Num(-5), Num(-6), Num(-7)],
 [Num(0),
  Num(-1),
  Num(-2),
  Num(-3),
  Num(-4),
  Num(-5),
  Num(-6),
  Num(-7),
  Num(-8)],
 [Num(0),
  Num(-1),
  Num(-2),
  Num(-3),
  Num(-4),
  Num(-5),
  Num(-6),
  Num(-7),
  Num(-8),
  Num(-9)],
 [Num(0),
  Num(-1),
  Num(-2),
  Num(-3),
  Num(-4),
  Num(-5),
  Num(-6),
  Num(-7),
  Num(-8),
  Num(-9),
  Num(-10)],
 [Num(0),
  Num(-1),
  Num(-2),
  Num(-3),
  Num(-4),
  Num(-5),
  Num(-6),
  Num(-7),
  Num(-8),
  Num(-9),
  Num(-10),
  Num(-11)],
 [Num(0),
  Num(-1),
  Num(-2),
  Num(-3),
  Num(-4),
  Num(-5),
  Num(-6),
  Num(-7),
  Num(-8),
  Num(-9),
  Num(-10),
  Num(-11),
  Num(-12)],
 [Num(0),
  Num(-1),
  Num(-2),
  Num(-3),
  Num(

In [225]:
inverted.visualize()

CytoscapeWidget(cytoscape_layout={'name': 'dagre', 'rankDir': 'BT', 'nodeSep': 10, 'edgeSep': 10, 'spacingFact…

In [227]:
result = objs.map(lambda x: x + 5).compute()
result

[[Num(0), Num(1), Num(2), Num(3), Num(4)],
 [Num(0), Num(1), Num(2), Num(3), Num(4), Num(5)],
 [Num(0), Num(1), Num(2), Num(3), Num(4), Num(5), Num(6)],
 [Num(0), Num(1), Num(2), Num(3), Num(4), Num(5), Num(6), Num(7)],
 [Num(0), Num(1), Num(2), Num(3), Num(4), Num(5), Num(6), Num(7), Num(8)],
 [Num(0),
  Num(1),
  Num(2),
  Num(3),
  Num(4),
  Num(5),
  Num(6),
  Num(7),
  Num(8),
  Num(9)],
 [Num(0),
  Num(1),
  Num(2),
  Num(3),
  Num(4),
  Num(5),
  Num(6),
  Num(7),
  Num(8),
  Num(9),
  Num(10)],
 [Num(0),
  Num(1),
  Num(2),
  Num(3),
  Num(4),
  Num(5),
  Num(6),
  Num(7),
  Num(8),
  Num(9),
  Num(10),
  Num(11)],
 [Num(0),
  Num(1),
  Num(2),
  Num(3),
  Num(4),
  Num(5),
  Num(6),
  Num(7),
  Num(8),
  Num(9),
  Num(10),
  Num(11),
  Num(12)],
 [Num(0),
  Num(1),
  Num(2),
  Num(3),
  Num(4),
  Num(5),
  Num(6),
  Num(7),
  Num(8),
  Num(9),
  Num(10),
  Num(11),
  Num(12),
  Num(13)],
 [Num(0),
  Num(1),
  Num(2),
  Num(3),
  Num(4),
  Num(5),
  Num(6),
  Num(7),
  Num(8),
